# Online Retail Association (Market Basket Analysis)

This notebook performs **association rule mining** on the Online Retail dataset.

We will:
1. Clean the dataset (remove missing CustomerID, duplicates, invalid quantities)
2. Restrict analysis to **one country**
3. Convert invoice data into **transaction (basket) format**
4. Generate and interpret rules using **support, confidence, and lift**

In [69]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pandas as pd
import numpy as np
from itertools import combinations

## 1. Load Dataset

In [70]:
# Load the Online Retail dataset
# Using the CSV file provided with the assignment

data_path = "mba-data.csv"
df = pd.read_csv(data_path, encoding="ISO-8859-1", sep=";")

# Preview the data
df.shape
df.head()
df.columns

/var/folders/x_/jmz5m6dj5yjg90n8zy10rn2dn_6lwp/T/ipykernel_17714/2430994280.py:5: DtypeWarning: Columns (0: ï»¿BillNo) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path, encoding="ISO-8859-1", sep=";")


(522064, 7)

,ï»¿BillNo,Itemname,Quantity,Date,Price,CustomerID,Country
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,01.12.2010 08:26,"2,55",17850.0,United Kingdom
1,536365,WHITE METAL LANTERN,6,01.12.2010 08:26,"3,39",17850.0,United Kingdom
2,536365,CREAM CUPID HEARTS COAT HANGER,8,01.12.2010 08:26,"2,75",17850.0,United Kingdom
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6,01.12.2010 08:26,"3,39",17850.0,United Kingdom
4,536365,RED WOOLLY HOTTIE WHITE HEART.,6,01.12.2010 08:26,"3,39",17850.0,United Kingdom


Index(['ï»¿BillNo', 'Itemname', 'Quantity', 'Date', 'Price', 'CustomerID',
       'Country'],
      dtype='str')

## 2. Data Cleaning

We remove:
- rows with missing CustomerID
- duplicate rows
- invalid quantities (Quantity <= 0)

These steps ensure only valid purchase transactions are used.

In [71]:
# Make a working copy
df_clean = df.copy()

# Remove rows with missing CustomerID
df_clean = df_clean.dropna(subset=["CustomerID"])

# Remove duplicate rows
df_clean = df_clean.drop_duplicates()

# Remove invalid quantities (returns or errors)
df_clean = df_clean[df_clean["Quantity"] > 0]

df_clean.shape

(382813, 7)

## 3. Restrict Analysis to One Country

We analyze only **United Kingdom** data to avoid mixing customer behavior across countries.

In [72]:
country = "United Kingdom"
df_country = df_clean[df_clean["Country"] == country].copy()

df_country.shape
df_country["Country"].value_counts()

(349188, 7)

Country
United Kingdom    349188
Name: count, dtype: int64

## 4. Convert Data into Transaction Format

Each **InvoiceNo** represents one transaction (basket).
We group all product descriptions bought in the same invoice.

In [73]:
# Fix column name first
df = df.rename(columns={"ï»¿BillNo": "BillNo"})

# Clean the dataset (basic required cleaning)
df_clean = df.dropna(subset=["CustomerID"]).drop_duplicates()
df_clean = df_clean[df_clean["Quantity"] > 0]

# Pick one country
country = "United Kingdom"   # change if needed
df_country = df_clean[df_clean["Country"] == country].copy()

# Clean item names
df_country["Itemname"] = df_country["Itemname"].astype(str).str.strip()

# Convert to transaction format (BillNo = transaction)
transactions = (
    df_country.groupby("BillNo")["Itemname"]
    .apply(lambda x: list(set(x)))
)

transactions.head(), len(transactions)


(BillNo
 536365    [RED WOOLLY HOTTIE WHITE HEART., WHITE METAL L...
 536366    [HAND WARMER UNION JACK, HAND WARMER RED POLKA...
 536367    [HOME BUILDING BLOCK WORD, ASSORTED COLOUR BIR...
 536368    [JAM MAKING SET WITH JARS, YELLOW COAT RACK PA...
 536369                           [BATH BUILDING BLOCK WORD]
 Name: Itemname, dtype: object,
 16651)

## 5. One-Hot Encode Transactions

We convert transactions into a boolean basket matrix suitable for Apriori.

In [74]:
# Get most frequent items to limit computation
top_items = transactions.explode().value_counts().head(50).index.tolist()

# Create basket matrix
basket = pd.DataFrame(False, index=transactions.index, columns=top_items)

for invoice, items in transactions.items():
    for item in items:
        if item in basket.columns:
            basket.loc[invoice, item] = True

basket.head()
basket.shape

,WHITE HANGING HEART T-LIGHT HOLDER,JUMBO BAG RED RETROSPOT,REGENCY CAKESTAND 3 TIER,ASSORTED COLOUR BIRD ORNAMENT,PARTY BUNTING,LUNCH BAG RED RETROSPOT,SET OF 3 CAKE TINS PANTRY DESIGN,LUNCH BAG BLACK SKULL.,PAPER CHAIN KIT 50'S CHRISTMAS,NATURAL SLATE HEART CHALKBOARD,...,ROSES REGENCY TEACUP AND SAUCER,PAPER CHAIN KIT VINTAGE CHRISTMAS,LUNCH BAG WOODLAND,PLEASE ONE PERSON METAL SIGN,HOME BUILDING BLOCK WORD,CHOCOLATE HOT WATER BOTTLE,RABBIT NIGHT LIGHT,RED HANGING HEART T-LIGHT HOLDER,GIN + TONIC DIET METAL SIGN,6 RIBBONS RUSTIC CHARM
BillNo,,,,,,,,,,,,,,,,,,,,,
536365,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536366,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536367,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
536368,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536369,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


(16651, 50)

## 6. Apriori: Frequent Itemsets

**Support** = fraction of transactions containing an itemset

In [75]:
from itertools import combinations

def apriori_itemsets(basket, min_support=0.02, max_len=3):
    """
    Generate frequent itemsets up to length max_len using a simple Apriori-style search.
    basket: boolean DataFrame (rows=transactions, cols=items)
    """
    n = len(basket)
    support_dict = {}

    for item in basket.columns:
        supp = basket[item].mean()
        if supp >= min_support:
            support_dict[frozenset([item])] = supp

    items = list(basket.columns)

    for k in range(2, max_len + 1):
        # generate candidate combinations of items of size k
        for combo in combinations(items, k):
            combo = frozenset(combo)

            # compute support: invoices where all items are True
            supp = basket[list(combo)].all(axis=1).mean()

            if supp >= min_support:
                support_dict[combo] = supp

    # Convert to DataFrame
    freq_itemsets = pd.DataFrame(
        [{"itemset": k, "support": v, "length": len(k)} for k, v in support_dict.items()]
    ).sort_values(["length", "support"], ascending=[True, False])

    return freq_itemsets, support_dict


freq_itemsets, support_dict = apriori_itemsets(basket, min_support=0.01, max_len=3)

freq_itemsets.head(10), freq_itemsets["length"].value_counts()


(                                           itemset   support  length
 0  frozenset({WHITE HANGING HEART T-LIGHT HOLDER})  0.113146       1
 1             frozenset({JUMBO BAG RED RETROSPOT})  0.086902       1
 2            frozenset({REGENCY CAKESTAND 3 TIER})  0.084680       1
 3       frozenset({ASSORTED COLOUR BIRD ORNAMENT})  0.078073       1
 4                       frozenset({PARTY BUNTING})  0.077533       1
 5             frozenset({LUNCH BAG RED RETROSPOT})  0.067263       1
 6    frozenset({SET OF 3 CAKE TINS PANTRY DESIGN})  0.060477       1
 7             frozenset({LUNCH BAG  BLACK SKULL.})  0.059816       1
 8      frozenset({PAPER CHAIN KIT 50'S CHRISTMAS})  0.056753       1
 9      frozenset({NATURAL SLATE HEART CHALKBOARD})  0.056333       1,
 length
 2    97
 1    50
 3    45
 Name: count, dtype: int64)

## 7. Generate Association Rules

Metrics:
- **Confidence**: likelihood of consequent given antecedent
- **Lift**: strength of association (>1 = positive relationship)

In [76]:
def generate_rules_from_support(support_dict, min_conf=0.3):
    rules = []

    for itemset, supp_itemset in support_dict.items():
        # rules need at least 2 items
        if len(itemset) < 2:
            continue

        # try all possible antecedents
        for r in range(1, len(itemset)):
            for antecedent in combinations(list(itemset), r):
                antecedent = frozenset(antecedent)
                consequent = itemset - antecedent

                supp_a = support_dict.get(antecedent)
                supp_b = support_dict.get(consequent)

                # if we don't know support(B), skip
                if supp_a is None or supp_b is None:
                    continue

                confidence = supp_itemset / supp_a
                lift = confidence / supp_b

                if confidence >= min_conf:
                    rules.append({
                        "Antecedent": antecedent,
                        "Consequent": consequent,
                        "Support": supp_itemset,
                        "Confidence": confidence,
                        "Lift": lift
                    })

    return pd.DataFrame(rules)


rules = generate_rules_from_support(support_dict, min_conf=0.3)

# Always check empty before sorting (prevents KeyError)
if rules.empty:
    print("No rules found. Lower min_support/min_conf or increase top items.")
else:
    rules.sort_values("Lift", ascending=False).head(10)

,Antecedent,Consequent,Support,Confidence,Lift
78,frozenset({ALARM CLOCK BAKELIKE GREEN}),frozenset({ALARM CLOCK BAKELIKE RED}),0.027266,0.657971,14.453661
79,frozenset({ALARM CLOCK BAKELIKE RED}),frozenset({ALARM CLOCK BAKELIKE GREEN}),0.027266,0.598945,14.453661
94,"frozenset({WHITE HANGING HEART T-LIGHT HOLDER,...",frozenset({WOODEN PICTURE FRAME WHITE FINISH}),0.010690,0.712000,13.882333
229,"frozenset({LUNCH BAG SPACEBOY DESIGN, LUNCH BA...",frozenset({LUNCH BAG WOODLAND}),0.010930,0.544910,13.461869
157,"frozenset({LUNCH BAG SPACEBOY DESIGN, LUNCH BA...",frozenset({LUNCH BAG WOODLAND}),0.013393,0.543902,13.436973
155,frozenset({LUNCH BAG WOODLAND}),"frozenset({LUNCH BAG SPACEBOY DESIGN, LUNCH BA...",0.013393,0.330861,13.436973
214,"frozenset({LUNCH BAG SPACEBOY DESIGN, LUNCH BA...",frozenset({LUNCH BAG WOODLAND}),0.011711,0.537190,13.271145
202,"frozenset({LUNCH BAG BLACK SKULL., LUNCH BAG ...",frozenset({LUNCH BAG SUKI DESIGN}),0.011651,0.642384,12.794662
224,"frozenset({LUNCH BAG APPLE DESIGN, LUNCH BAG C...",frozenset({LUNCH BAG SUKI DESIGN}),0.010150,0.630597,12.559893
93,"frozenset({WOODEN PICTURE FRAME WHITE FINISH, ...",frozenset({WOODEN FRAME ANTIQUE WHITE}),0.010690,0.583607,12.394940


## 8. Interpretation of Rules

- **Support** indicates how frequently an itemset appears in the entire dataset.
  A higher support value means the combination of items is commonly purchased together.

- **Confidence** measures the likelihood that the consequent is purchased
  when the antecedent has already been purchased.
  A high confidence value implies that customers who buy the antecedent
  often also buy the consequent.

- **Lift** measures the strength of the association between items compared
  to random chance.
  A lift value greater than 1 indicates a positive association,
  meaning the items occur together more often than expected.

These association rules can be used for product placement,
cross-selling strategies, and recommendation systems in retail environments.
